Data Link: [ASF](https://search.asf.alaska.edu/#/?maxResults=250&zoom=8.480&center=140.971,36.244&polygon=POLYGON((140.6718%2036.8047,140.8329%2036.8047,140.8329%2036.9264,140.6718%2036.9264,140.6718%2036.8047))&dataset=ALOS&start=2011-02-28T15:00:00Z&end=2011-04-19T14:59:59Z&productTypes=L1.1&resultsLoaded=true&granule=ALPSRP271970730-L1.1&polarizations=HH&flightDirs=Ascending&beamModes=FBS)

## 読み込み

In [ ]:
import os
import numpy as np
import warnings
import tifffile
import cv2

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

In [ ]:
PATH_ROOT_ALOS_M = '../data/higashinihon/ALPSRP271970730-H1.1__A/'
PATH_ROOT_ALOS_S = '../data/higashinihon/ALPSRP278680730-H1.1__A/'

PATH_OUTPUT = os.path.join('output', 'ALOS_RE-shift_HIGASHI_NIHON2')
os.makedirs(PATH_OUTPUT, exist_ok=True)

In [ ]:
from utils.ceos_io import ALOSPALSARData

alos_m = ALOSPALSARData(PATH_ROOT_ALOS_M)
alos_s = ALOSPALSARData(PATH_ROOT_ALOS_S)

slc_m = alos_m.read_slc()
slc_s = alos_s.read_slc()

In [ ]:
plt.figure(figsize=(16, 12), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.title('ALOS PALSAR Intencity', fontsize=12, fontweight='bold')
plt.imshow(np.abs(slc_m[0]), cmap='gray')
plt.colorbar(shrink=0.6)
plt.subplot(1, 2, 2)
plt.title('ALOS PALSAR Phase', fontsize=12, fontweight='bold')
plt.imshow(np.angle(slc_m[0]), cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='nearest')
plt.colorbar(shrink=0.6)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m.png'))
plt.show();
plt.clf()
plt.close()

In [ ]:
pos_m = alos_m.read_location()
pos_s = alos_s.read_location()
print('ALOS `M`ain Path 4 edgs points:  \n', np.array(pos_m).reshape(4, 2), )

In [ ]:
alos_m.write_single_look_complex(
    PATH_GEOTIF_SLC=os.path.join(PATH_OUTPUT, 'ALOS_m.tif'), 
    imgs=slc_m)

In [ ]:
bbox = [
    140.8, 37.2,  # top left
    141, 36.9,  # bottom right
]

slc_m_crop = alos_m.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_HH_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_HH.tif')
)
slc_m_crop.shape

In [ ]:
plt.figure(figsize=(16, 12), dpi=160, facecolor='w', edgecolor='k')

plt.title('ALOS PALSAR Crop Area Intencity', fontsize=12, fontweight='bold')
plt.imshow(np.abs(slc_m_crop), cmap='gray', )
plt.colorbar(shrink=0.8)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m_crop_intencity.png'))
plt.show();
plt.clf()
plt.close()

## 軌道解析

In [ ]:
(orb_m, obs_m) = alos_m.read_orbit()
(orb_s, obs_s) = alos_s.read_orbit()

plt.figure(figsize=(10, 4), dpi=80, facecolor='w', edgecolor='k')
plt.plot(obs_m, orb_m[:, 0], label='Orbit X')
plt.plot(obs_m, orb_m[:, 1], label='Orbit Y')
plt.plot(obs_m, orb_m[:, 2], label='Orbit Z')
plt.title('ALOS PALSAR ECEF Orbit', fontsize=12, fontweight='bold')
plt.xlabel('Milli Sec (Time)')
plt.ylabel('Height (m)')
plt.legend()
plt.grid()
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m_orbit.png'))
plt.show();
plt.clf()
plt.close()

for idx_axis, axis in enumerate(['X', 'Y', 'Z']):
    plt.figure(figsize=(8, 3), dpi=80, facecolor='w', edgecolor='k')
    plt.plot(obs_m, orb_m[:, idx_axis], label=f'Orbit {axis}')
    plt.title(f'ALOS PALSAR ECEF Orbit {axis}', fontsize=12, fontweight='bold')
    plt.xlabel('Milli Sec (Time)')
    plt.ylabel('Height (m)')
    plt.grid()
    plt.tight_layout(pad=1.3)
    plt.savefig(os.path.join(PATH_OUTPUT, f'alos_m_orbit_{axis}.png'))
    plt.show();
    plt.clf()
    plt.close()

In [ ]:
(A, B, CS, CL) = alos_m.read_geo_matrix()
print('ALOS `M`ain Path Geo Matrix:  \n', A.shape, B.shape, CS, CL)


In [ ]:
xyz_m_loc, (idx_sample, idx_line) = alos_m.read_xyz_raster(A, B, CS, CL)
print('ALOS `M`ain Path Geo Obsebasion Area Raster:  \n', xyz_m_loc.shape)

In [ ]:
from utils.interferogram import get_slant_range

slant_m = get_slant_range(orb_m, xyz_m_loc, idx_line)

In [ ]:
plt.figure(figsize=(7, 10), dpi=70, facecolor='w', edgecolor='k')
plt.title('ALOS PALSAR Slant Range', fontsize=12, fontweight='bold')
plt.imshow(slant_m, cmap='autumn',)
plt.colorbar(shrink=0.8)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_m_slant_range.png'))
plt.show();
plt.clf()
plt.close()

## Sub Path

In [ ]:
from utils.interferogram import (
    interferogram, 
    orbital_stripe, 
    wrap_phase, 
    coregistoration_homomorphpy_cfloat, 
    to8bit,
    coregistration_phase_only_correlation
    )

In [ ]:

alos_s.write_single_look_complex(
    PATH_GEOTIF_SLC=os.path.join(PATH_OUTPUT, 'ALOS_s.tif'), 
    imgs=slc_s)
slc_s_crop = alos_s.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_HH_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_HH.tif')
)
(orb_s, obs_s) = alos_s.read_orbit()
slant_s = get_slant_range(orb_s, xyz_m_loc, idx_line)
slant_m = get_slant_range(orb_m, xyz_m_loc, idx_line)
slant_s.shape, obs_m.shape, xyz_m_loc.shape, idx_line.shape

## オフセット処理

In [ ]:
RANGE_LEFT_PAD = 0
RANGE_RIGHT_PAD = 1
AZIMUTH_TOP_PAD = 0
AZIMUTH_BOTTOM_PAD = 0
RANGE_LEFT_CUT = 0
RANGE_RIGHT_CUT = 0
AZIMUTH_TOP_CUT = 0
AZIMUTH_BOTTOM_CUT = 0

shift_txt = f'P-R{RANGE_LEFT_PAD}-{RANGE_RIGHT_PAD}A{AZIMUTH_TOP_PAD}-{AZIMUTH_BOTTOM_PAD}'
shift_txt += f'_C-R{RANGE_LEFT_CUT}-{RANGE_RIGHT_CUT}A{AZIMUTH_TOP_CUT}-{AZIMUTH_BOTTOM_CUT}'
print(shift_txt)

H_S, W_S = slc_s[0].shape

# shift flow
slc_s_shift = np.pad(
    slc_s[0][
    AZIMUTH_TOP_CUT:H_S-AZIMUTH_BOTTOM_CUT,
    RANGE_LEFT_CUT:W_S-RANGE_RIGHT_CUT
    ],
    (
        (AZIMUTH_TOP_PAD, AZIMUTH_BOTTOM_PAD), 
        (RANGE_LEFT_PAD, RANGE_RIGHT_PAD)
        ),
    'constant',
    constant_values=0
    )

print(slc_s_shift.shape)

# recrop flow
alos_s.write_single_look_complex(
    PATH_GEOTIF_SLC=os.path.join(PATH_OUTPUT, 'ALOS_s_shift.tif'), 
    imgs=slc_s_shift,
    )

slc_s_shift_crop = alos_s.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_shift_HH_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_s_shift_HH.tif')
)

## レジストレーション

In [ ]:
_, slc_s_crop_cor ,difference  = coregistration_phase_only_correlation(
        slc_m_crop, slc_s_shift_crop,
        s_left_pad=0, s_right_pad=1, s_top_pad=0, s_bottom_pad=0,
        s_left_cut=0, s_right_cut=0, s_top_cut=0, s_bottom_cut=0
        
    )

print('ALOS `S`ub Path Coregistration Difference:', difference)
print(f'Round Pixel : (H x W) = {round(difference[0])}, {round(difference[1])}')
print(f'Size of Main Path: {slc_m_crop.shape}')
print(f'Size of Sub  Path: {slc_s_shift_crop.shape} -> {slc_s_crop_cor.shape}')

ifgm = interferogram(slc_m_crop, slc_s_crop_cor)

amp_s_crop_cor = to8bit(np.abs(slc_s_crop_cor))
amp_m_crop = to8bit(np.abs(slc_m_crop))

amp_ms_crop = np.stack([to8bit(np.abs(slc_m_crop)), amp_s_crop_cor, amp_s_crop_cor], axis=2)
cv2.imwrite(os.path.join(PATH_OUTPUT, f'amp_ms_crop_{shift_txt}.png'), to8bit(amp_ms_crop[:,:,::-1]))

plt.figure(figsize=(14, 8), dpi=160, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.imshow(ifgm, 
           cmap='jet', 
        #    interpolation='nearest'
           )
plt.colorbar(shrink=0.6)

plt.title(f'ALOS PALSAR Interferogram', fontsize=12, fontweight='bold')

plt.subplot(1, 2, 2)
plt.title(f'ALOS PALSAR Overlay Intencity', fontsize=12, fontweight='bold')
plt.imshow(amp_ms_crop)
plt.colorbar(shrink=0.6)

plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, f'ifgm_fisrt_shift_{shift_txt}.png'))
plt.show();
plt.clf()
plt.close()

In [ ]:
plt.imsave(os.path.join(PATH_OUTPUT, f'ifgm-vis_{shift_txt}.png'), ifgm, cmap='hsv')

In [ ]:
phase_rgb = cv2.imread(os.path.join(PATH_OUTPUT, f'ifgm-vis_{shift_txt}.png'))

plt.figure(figsize=(10, 10), dpi=160, facecolor='w', edgecolor='k')
plt.imshow(phase_rgb, alpha=0.9)
plt.imshow(amp_ms_crop[:,:,0], alpha=0.3, cmap='gray')
plt.title(f'ALOS PALSAR Intencity & Interferogram', fontsize=12, fontweight='bold')
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, f'ifgm_overlay_{shift_txt}.png'))
plt.show();
plt.clf()
plt.close()

## 軌道縞 - 軌道差

In [ ]:
plt.figure(figsize=(7, 10), dpi=70, facecolor='w', edgecolor='k')
plt.title('ALOS PALSAR Slant Range', fontsize=12, fontweight='bold')
plt.imshow(slant_s, cmap='autumn',)
plt.colorbar(shrink=0.8)
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_s_slant_range.png'))
plt.show();
plt.clf()
plt.close()

In [ ]:
wave_length = alos_m.wave_length
print('Wave Length λ: ', wave_length)

stripe_orb = orbital_stripe(slant_m, slant_s, wave_length)

plt.figure(figsize=(6, 8), dpi=80, facecolor='w', edgecolor='k')
plt.imshow(stripe_orb, cmap='gist_rainbow', vmin=-np.pi, vmax=np.pi, interpolation='nearest')
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Orbital Stripe', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'alos_orbital_stripe.png'))
plt.show();
plt.clf()
plt.close()

In [ ]:
alos_s.write_geotiff(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'stripe_orbit.tif'), 
    img=stripe_orb,
)

In [ ]:
stripe_orb_crop = alos_s.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'stripe_orbit_crop.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'stripe_orbit.tif')
)

In [ ]:
plt.figure(figsize=(12, 8), dpi=70, facecolor='w', edgecolor='k')
plt.imshow(stripe_orb_crop, 
           cmap='jet', 
           vmin=-np.pi, vmax=np.pi, 
           )
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Orbital Stripe Crop Area', fontsize=12, fontweight='bold')
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'orbital_stripe_crop.png'))
plt.show();
plt.clf()
plt.close()

In [ ]:
ifgm_rm_orb = wrap_phase(stripe_orb_crop - ifgm)

plt.figure(figsize=(12, 10), dpi=160, facecolor='w', edgecolor='k')
plt.imshow(ifgm_rm_orb, 
           cmap='jet', 
           vmin=-np.pi, vmax=np.pi, 
         #   interpolation='nearest',
           )
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Interferogram Remove Orbital Stripe', fontsize=12, fontweight='bold')
plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_remove_orbital_stripe.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
plt.imsave(os.path.join(PATH_OUTPUT, 'ifgm-vis_remove_orbital_stripe.png'), ifgm_rm_orb, cmap='jet')

In [ ]:
phase_rgb = cv2.imread(os.path.join(PATH_OUTPUT, 'ifgm-vis_remove_orbital_stripe.png'))

plt.figure(figsize=(18,12), dpi=200, facecolor='w', edgecolor='k')
plt.title('Interferogrum Phase & Intency Filter')
plt.imshow(phase_rgb, alpha=0.95)
# plt.colorbar(shrink=0.5, aspect=20)
plt.imshow(amp_ms_crop, alpha=0.5,)
plt.tight_layout()
plt.savefig(PATH_OUTPUT + f'/ifgm-overlay_rm_orbit.png')
plt.show();
plt.clf();plt.close()

## 地形差 - 地形縞


In [ ]:
PATH_SRTM_DEM = '../data/higashinihon/srtm_dem_higashinihon_nogeo_modified_pol2_cub.tif'
PATH_SRTM_DEM_CUT = '../data/higashinihon/srtm_dem_higashinihon2_nogeo_modified_pol2_cub_cut.tif'

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
dem = tifffile.imread(PATH_SRTM_DEM_CUT)
plt.imshow(dem, cmap='terrain', vmin=0)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM Mod CUT', fontsize=12, fontweight='bold')
plt.tight_layout(pad=1.1)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.savefig(os.path.join(PATH_OUTPUT, 'dem_srtm_mod_cut.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
PATH_DEM_CUT = PATH_SRTM_DEM_CUT
dem = tifffile.imread(PATH_DEM_CUT).astype(np.float32)
dem = np.where(dem < 0, 0, dem) # preprocess

H_DEM, W_DEM = dem.shape
print('Size of DEM: ', dem.shape)

dem = alos_m.resize_crop_area(dem)
H_DEM_RESIZE, W_DEM_RESIZE = dem.shape
print('Size of DEM Resize: ', dem.shape)


W_SCALE = W_DEM_RESIZE / W_DEM
print('Longitude Scale of DEM: ', W_SCALE)

plt.figure(figsize=(14, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(dem, cmap='terrain', vmin=0, )
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'dem.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
xyz_elev_m_loc, (_, idx_line_elev) = alos_m.read_xyz_raster(
    A, B, CS, CL, dem=dem, 
    bbox=bbox, PATH_OUT=PATH_OUTPUT)


In [ ]:
idx_line_elev_crop = alos_m.write_crop_geotif(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_line-elev.tif'), 
    img=idx_line_elev.astype(np.float32), bbox=bbox)

idx_line_elev_crop = idx_line_elev_crop.astype(np.int32)

In [ ]:
slant_elev_s_crop = get_slant_range(orb_s, xyz_elev_m_loc, idx_line_elev_crop)
slant_elev_m_crop = get_slant_range(orb_m, xyz_elev_m_loc, idx_line_elev_crop)

In [ ]:
from osgeo import gdal

In [ ]:
from utils.interferogram import slide_dem

dem_slide = slide_dem(slant_elev_m_crop, dem)

xyz_elev_slide_m_loc, (_, idx_line_elev_slide) = alos_m.read_xyz_raster(
    A, B, CS, CL, dem=dem_slide,
    bbox=bbox, PATH_OUT=PATH_OUTPUT
    )

idx_line_elev_slide_crop = alos_m.write_crop_geotif(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'ALOS_m_line-elev-slide.tif'), 
    img=idx_line_elev_slide.astype(np.float32), bbox=bbox)

idx_line_elev_slide_crop = idx_line_elev_slide_crop.astype(np.int32)

slant_elev_slide_s = get_slant_range(orb_s, xyz_elev_slide_m_loc, idx_line_elev_slide_crop)
slant_elev_slide_m = get_slant_range(orb_m, xyz_elev_slide_m_loc, idx_line_elev_slide_crop)

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(dem_slide, cmap='terrain', vmin=0, vmax=1117)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m Slide', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'dem_slide.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'dem_slide.tif'), dem_slide)

In [ ]:
plt.figure(figsize=(8, 6), dpi=70, facecolor='w', edgecolor='k')
plt.imshow(slant_elev_m_crop, cmap='spring',)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Slant Range from DEM', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'slant_elev_m.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
alos_m.write_geotiff(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_m.tif'), 
    img=slant_m,
    gdal_type=gdal.GDT_Float64
)
slant_m_cut = alos_m.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_m_cut.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_m.tif')
)
alos_m.write_geotiff(
    PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_s.tif'), 
    img=slant_s,
    gdal_type=gdal.GDT_Float64
)
slant_s_cut = alos_m.crop_geotiff(
    bbox,
    PATH_OUTPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_s_cut.tif'), 
    PATH_INPUT_GEOTIF=os.path.join(PATH_OUTPUT, 'slant_s.tif')
)

In [ ]:
plt.figure(figsize=(10, 6), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(slant_m_cut - slant_elev_m_crop, cmap='gist_earth',)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Slant Range from DEM', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'slant_diff_m.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
from utils.interferogram import get_topography

topography = get_topography(
    slant_m_cut, slant_s_cut,
    slant_elev_slide_m, slant_elev_slide_s, 
    wave_length)

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(topography, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Topography Phase', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'topography.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
ifgm_topo = wrap_phase(topography - ifgm_rm_orb)

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(ifgm_topo, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Interferogram removal Topography Phase', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topography.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
plt.imsave(os.path.join(PATH_OUTPUT, 'ifgm-vis_topography.png'), ifgm_topo, cmap='hsv')

In [ ]:
phase_rgb_topo = cv2.imread(os.path.join(PATH_OUTPUT, 'ifgm-vis_topography.png'))

plt.figure(figsize=(18,12), dpi=200, facecolor='w', edgecolor='k')
plt.title('Interferogrum Phase & Intency')
plt.imshow(phase_rgb_topo, alpha=0.95)
# plt.colorbar(shrink=0.5, aspect=20)
plt.imshow(amp_ms_crop, alpha=0.75,)
plt.savefig(PATH_OUTPUT + f'/ifgm-overlay_topography.png')
plt.show();
plt.clf();plt.close()


In [ ]:
from utils.interferogram import goldshtein_phase_filter_sliding_window

In [ ]:
clx_rm_topo = amp_m_crop * np.exp(1j * ifgm_topo)

slc_rm_topo_filter = goldshtein_phase_filter_sliding_window(
    clx_rm_topo, 
    # alpha=0.85, patch_size=32, step=1, filter_size=7
    )
ifgm_topo_filtered = np.angle(slc_rm_topo_filter)

plt.figure(figsize=(16, 10), dpi=200, facecolor='w', edgecolor='k')
plt.imshow(ifgm_topo_filtered, cmap='hsv', vmin=-np.pi, vmax=np.pi,)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Interferogram removal Topography Phase Filtered', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topography_filtered.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
plt.imsave(os.path.join(PATH_OUTPUT, 'ifgm-vis_topography_filtered.png'), ifgm_topo_filtered, cmap='hsv')

In [ ]:
phase_rgb_topo = cv2.imread(os.path.join(PATH_OUTPUT, 'ifgm-vis_topography_filtered.png'))

plt.figure(figsize=(14 ,12), dpi=200, facecolor='w', edgecolor='k')
plt.title('Interferogrum Phase & Intency Filter')
plt.imshow(phase_rgb_topo, alpha=0.9)
plt.imshow(amp_ms_crop, alpha=0.7,)
plt.savefig(PATH_OUTPUT + f'/ifgm-overlay_topography_filter.png')
plt.show();
plt.clf();plt.close()

## コヒーレンス

In [ ]:
from utils.interferogram import coherence_multi_process

PATCH_SIZE = 2

coherence = coherence_multi_process(slc_m_crop, slc_s_crop_cor, PATCH_SIZE)

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(coherence, cmap='gnuplot', vmin=0, vmax=1)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Coherence', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence.png'))
plt.show();
plt.clf();plt.close()

tifffile.imwrite(os.path.join(PATH_OUTPUT, 'coherence.tif'), coherence)

In [ ]:
th_coherence = 0.3

# plot histogram
plt.figure(figsize=(12, 4), facecolor='w', edgecolor='k')
plt.hist(coherence.ravel(), bins=200, color='b', range=(-0.05, 1.05), label='Coherence')
plt.title('Coherence Histogram', fontsize=12, fontweight='bold')
plt.grid()
plt.vlines(th_coherence, 0, 800000, 
           colors='red', linestyle='dashed', linewidth=3,
           label='Weak Threshold')
plt.tight_layout()
plt.legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence_hist.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
coherence_clip = np.where(coherence > th_coherence, coherence, 0)

# plot histogram
plt.figure(figsize=(12, 4), facecolor='w', edgecolor='k')
plt.hist(coherence_clip.ravel(), bins=200, color='b', range=(-0.05, 1.05), label='Coherence')
plt.title('Coherence Cut Histogram', fontsize=12, fontweight='bold')
plt.grid()
plt.vlines(th_coherence, 0, 800000, 
           colors='red', linestyle='dashed', linewidth=3,
           label='Weak Threshold')
plt.tight_layout()
plt.legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence_cut_hist.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(coherence_clip, cmap='gnuplot', vmin=0, vmax=1)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Coherence Cut', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'coherence_cut.png'))
plt.show();
plt.clf();plt.close()

## アンラップ

In [ ]:
import snaphu

In [ ]:
unw, conncomp = snaphu.unwrap(slc_rm_topo_filter, coherence,
                              nlooks=32.0, cost="smooth", init="mcf", 
                              nproc=8)

In [ ]:
num_plot_cycle = 10

plt.figure(figsize=(24, 8), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.imshow(unw, cmap='coolwarm', vmin=-np.pi*num_plot_cycle, vmax=np.pi*num_plot_cycle)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Unwrap Phase', fontsize=12, fontweight='bold')

plt.subplot(1, 2, 2)
# mask from dem
is_island = np.where(dem == 0, 0, 1)
unw_island = unw * is_island
plt.imshow(unw_island, cmap='bwr', vmin=-np.pi*num_plot_cycle, vmax=np.pi*num_plot_cycle)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Unwrap Phase DEM Masked', fontsize=12, fontweight='bold')

plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'unw.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
plt.figure(figsize=(56, 12), dpi=160)
plt.subplot(151)
plt.title('Unwrapped phase', fontsize=14, fontweight='bold')
plt.imshow(unw, cmap="plasma")
plt.colorbar(shrink=0.5)

plt.subplot(152)
plt.title('Original phase', fontsize=14, fontweight='bold')
plt.imshow(np.angle(slc_rm_topo_filter), cmap="hsv", vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.5)

plt.subplot(153)
plt.title('Coherence', fontsize=14, fontweight='bold')
plt.imshow(coherence, cmap="gray", vmin=th_coherence, vmax=1)
plt.colorbar(shrink=0.5)

plt.subplot(154)
plt.title('Amplitude Difference', fontsize=14, fontweight='bold')
plt.imshow(amp_ms_crop)
plt.colorbar(shrink=0.5)

plt.subplot(155)
plt.title('Connected components', fontsize=14, fontweight='bold')
plt.imshow(conncomp, cmap="Set1")
plt.colorbar(shrink=0.5)

plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'unwrapped_phase_info.png'))
plt.show()
plt.clf();plt.close()


## 出力保存


In [ ]:
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap.tif'), unw)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'unwrap_island.tif'), unw_island)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'connected_component.tif'), conncomp)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'coherence.tif'), coherence)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'amplitude_difference_3ch.tif'), amp_ms_crop)

# interferogram phase
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'ifgm_remove_orbital_stripe.tif'), ifgm_rm_orb)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'ifgm_topography.tif'), ifgm_topo)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'ifgm_topography_filtered.tif'), ifgm_topo_filtered)
tifffile.imwrite(os.path.join(PATH_OUTPUT, 'topography.tif'), topography)

## DInSAR

In [ ]:
difference_slant = np.divide(unw_island * wave_length, (np.square(2) * np.pi))

tifffile.imwrite(os.path.join(PATH_OUTPUT, 'difference_slant.tif'), difference_slant)

plt.figure(figsize=(24, 8), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 3, 1)
plt.imshow(difference_slant, cmap='coolwarm', vmin=-0.1, vmax=0)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Difference Slant -0.1 ~ 0 [m]', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 2)
plt.imshow(difference_slant, cmap='coolwarm', vmin=0, vmax=0.2)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Difference Slant 0 ~ 0.2 [m]', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 3)
plt.imshow(difference_slant, cmap='coolwarm', vmin=0.2, vmax=0.6)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Difference Slant 0.2 ~ 0.6 [m]', fontsize=12, fontweight='bold')

plt.tight_layout(pad=0.6)
plt.savefig(os.path.join(PATH_OUTPUT, 'difference_slant.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
from utils.interferogram import get_angle_vectors

def get_incident_angle(xyz_elev_loc, orb, idx_line):
    
    orb_idx = orb[idx_line, :]
    sla = orb_idx - xyz_elev_loc
    incident_angle = get_angle_vectors(sla, xyz_elev_loc)
    return incident_angle

incident_angle = get_incident_angle(xyz_elev_m_loc, orb=orb_m, idx_line=idx_line_elev_crop)

In [ ]:
plt.figure(figsize=(10, 6), dpi=70, facecolor='w', edgecolor='k')
plt.imshow(np.degrees(incident_angle), cmap='YlGnBu',
        #    vmin=20, vmax=70
           )
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Incident Angle [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'incident_angle.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
options = gdal.DEMProcessingOptions(
    slopeFormat='degree',
    )
gdal.DEMProcessing(
    os.path.join(PATH_OUTPUT, 'dem_slide_slope.tif'), 
    os.path.join(PATH_OUTPUT, 'dem_slide.tif'), 
    'slope', options=options)

slope = tifffile.imread(os.path.join(PATH_OUTPUT, 'dem_slide_slope.tif'))

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(slope, cmap='Oranges', vmin=5, vmax=85)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m Slide Slope [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'slope.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
slope_range_tan_minus = (np.roll(dem_slide, 1, axis=1) - dem_slide + 0.0001) * (30 / W_SCALE)
slope_range_tan_pulas = (dem_slide - np.roll(dem_slide, 1, axis=1) + 0.0001) * (30 / W_SCALE)
slope_range = np.degrees(np.arctan(slope_range_tan_pulas))/2 - np.degrees(np.arctan(slope_range_tan_minus))/2
# slope_range[slope_range == 0] = np.nan

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(slope_range, cmap='Spectral',)
plt.colorbar(shrink=0.8)
plt.title('SRTM DEM 30m Slide Slope Range [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'slope_range.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
local_incident_angle = np.degrees(incident_angle) - slope_range
# local_incident_angle[dem_slide == 0] = np.nan

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(local_incident_angle, cmap='brg', vmin=1, vmax=89)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Local Incident Angle [°]', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'local_incident_angle.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
# line-of-sight ==  difference of slant

difference_h = difference_slant * np.cos(np.radians(local_incident_angle))
difference_w = difference_slant * np.sin(np.radians(local_incident_angle))

diff_range = 0.20

plt.figure(figsize=(20, 8), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
plt.imshow(difference_w, cmap='PuOr', vmin=-diff_range, vmax=diff_range)
plt.colorbar(shrink=0.8)
plt.title(f'ALOS PALSAR Difference Width -{diff_range} ~ {diff_range} [m]', fontsize=12, fontweight='bold')

plt.subplot(1, 2, 2)
plt.imshow(difference_h, cmap='PRGn', vmin=-diff_range, vmax=diff_range)
plt.colorbar(shrink=0.8)
plt.title(f'ALOS PALSAR Difference Height -{diff_range} ~ {diff_range} [m]', fontsize=12, fontweight='bold')

plt.tight_layout(pad=0.6)
plt.savefig(os.path.join(PATH_OUTPUT, f'difference_wh_{diff_range}.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
slc_rm_orb = amp_m_crop * np.exp(1j * ifgm_rm_orb)

unw_topo = snaphu.unwrap(slc_rm_orb, coherence,
                              nlooks=32.0, cost="smooth", init="mcf", 
                              nproc=8)

In [ ]:
plt.figure(figsize=(24, 8), dpi=120, facecolor='w', edgecolor='k')
plt.subplot(1, 3, 1)
plt.imshow(unw_topo[0], cmap='terrain',)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Unwrap Phase Topography', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 2)
plt.imshow(np.angle(slc_rm_orb), cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Original Phase Topography', fontsize=12, fontweight='bold')

plt.subplot(1, 3, 3)
plt.imshow(unw_topo[1], cmap='Set2')
plt.colorbar(shrink=0.8)
plt.title('ALOS PALSAR Connected Components Topography', fontsize=12, fontweight='bold')

plt.tight_layout(pad=1.2)
plt.savefig(os.path.join(PATH_OUTPUT, 'unw_topo.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
# 3D plot
fig = plt.figure(figsize=(18, 14), dpi=120, facecolor='w', edgecolor='k')
ax = fig.add_subplot(121, projection='3d')

unw_topo_surface =  np.divide(unw_topo[0] * wave_length, (np.square(2) * np.pi))

X = np.arange(0, unw_topo_surface.shape[1], 1)
Y = np.arange(0, unw_topo_surface.shape[0], 1)

X, Y = np.meshgrid(X, Y)

surf = ax.plot_surface(X, Y, unw_topo_surface, cmap='twilight', rstride=3, cstride=3)
# color bar
fig.colorbar(surf)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_zlabel('Elevation [m]')
ax.view_init(elev=20, azim=-80, roll=0)

ax.set_title('Unwrap Phase Topography [m]', fontsize=16, fontweight='bold')

plt.savefig(os.path.join(PATH_OUTPUT, 'unw_topo_3d.png'))
plt.show();
plt.clf();plt.close()


## Multi Look

In [ ]:

from tqdm import tqdm
def multi_look(img, look_range=2, look_azimuth=2):
    H, W = img.shape
    H_ML = H // look_azimuth
    W_ML = W // look_range
    
    img_ml = np.zeros((H_ML, W_ML), dtype=img.dtype)
    
    for i in tqdm(range(H_ML), desc='Multi Looking ...', ncols=80, total=H_ML):
        for j in range(W_ML):
            img_ml[i, j] = np.mean(img[i*look_azimuth:(i+1)*look_azimuth, j*look_range:(j+1)*look_range])
    
    return img_ml

look_range = 2
look_azimuth = 2

ifgm_topo_filtered_ml = multi_look(ifgm_topo_filtered, look_range, look_azimuth)

plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
plt.imshow(ifgm_topo_filtered_ml, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(shrink=0.8)
plt.title('DInSAR Phase Filtered Multi Look', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(
    PATH_OUTPUT, 
    f'ifgm_topography_filtered_ml{look_range}-{look_azimuth}.png')
            )
plt.show();
plt.clf();plt.close()

plt.imsave(os.path.join(
    PATH_OUTPUT, 
    f'ifgm-vis_topography_filtered_ml{look_range}-{look_azimuth}.png'), 
    ifgm_topo_filtered_ml, cmap='hsv')


## Sectional View

In [ ]:
# sectional view

line_section = 1000
line_span    = [1500, 1800]

diff_line = difference_slant[line_section, line_span[0]:line_span[1]]
dem_line  = dem_slide[line_section, line_span[0]:line_span[1]]
slope_line = slope[line_section, line_span[0]:line_span[1]]

incident_angle_line = incident_angle[line_section, line_span[0]:line_span[1]]
local_incident_angle_line = local_incident_angle[line_section, line_span[0]:line_span[1]]

ifgm_topo_filtered_line = ifgm_topo_filtered[line_section, line_span[0]:line_span[1]]
coh_line = coherence[line_section, line_span[0]:line_span[1]]


plt.figure(figsize=(18, 18), facecolor='w', edgecolor='k')
plt.subplot(1, 3, 1)
plt.title('DEM Slide Sectional View', fontsize=12, fontweight='bold')
plt.imshow(dem_slide, cmap='terrain', vmin=0, vmax=600)
plt.plot([line_span[0], line_span[1]],[line_section, line_section],
         color='r', lw=4, label='Sectional Span')
plt.legend()

plt.subplot(1, 3, 2)
plt.title('Difference Slat Sectional View', fontsize=12, fontweight='bold')
plt.imshow(difference_slant, cmap='coolwarm',)
plt.plot([line_span[0], line_span[1]],[line_section, line_section],
         label='Sectional Span',
         color='g', lw=4 )
plt.legend()

plt.subplot(1, 3, 3)
plt.title('Phase Sectional View', fontsize=12, fontweight='bold')
plt.imshow(ifgm_topo_filtered, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.plot([line_span[0], line_span[1]],[line_section, line_section],
         label='Sectional Span', color='Black', lw=4)

plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_line.png'))
plt.show()
plt.clf();plt.close()

In [ ]:

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_line, label='DEM [m]', )
# plt.ylim(0, 50)
plt.quiver(
    np.arange(len(diff_line)), 
    dem_line, 
    np.zeros_like(diff_line), 
    -diff_line, 
    color='r', label='Difference', scale_units='xy',
    pivot = 'mid', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

# plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_difference.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
incident_angle_line_x = np.cos(incident_angle_line)
incident_angle_line_y = -np.sin(incident_angle_line)

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_line, label='DEM [m]')
# plt.ylim(0, 50)
plt.quiver(
    np.arange(len(diff_line)), 
    dem_line, 
    incident_angle_line_x, 
    incident_angle_line_y, 
    color='g', label='Incident Angle', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_incident_angle.png'))
plt.show();
plt.clf();plt.close()

In [ ]:
local_incident_angle_line[:6]

In [ ]:
local_incident_angle_line_x, local_incident_angle_line_y = np.cos(np.radians(90 - local_incident_angle_line)), -np.sin(np.radians(local_incident_angle_line))

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_line, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_line)), 
    dem_line, 
    local_incident_angle_line_x, 
    local_incident_angle_line_y, 
    color='b', label='Local Incident Angle', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004,
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_incident_angle.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
ifgm_line_x, ifgm_line_y = np.cos(ifgm_topo_filtered_line), -np.sin(ifgm_topo_filtered_line)

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_line, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_line)), 
    dem_line, 
    ifgm_line_x, 
    ifgm_line_y, 
    color='Black', label='Phase', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004,
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_phase.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
# sectional view 
range_section = 1600
range_span    = [900, 1100]

diff_range = difference_slant[range_span[0]:range_span[1], range_section]
dem_range  = dem_slide[range_span[0]:range_span[1], range_section]
slope_range = slope[range_span[0]:range_span[1], range_section]

incident_angle_range = incident_angle[range_span[0]:range_span[1], range_section]
local_incident_angle_range = local_incident_angle[range_span[0]:range_span[1], range_section]

ifgm_topo_filtered_range = ifgm_topo_filtered[range_span[0]:range_span[1], range_section]
coh_range = coherence[range_span[0]:range_span[1], range_section]

plt.figure(figsize=(18, 18), facecolor='w', edgecolor='k')
plt.subplot(1, 3, 1)
plt.title('DEM Slide Sectional View', fontsize=12, fontweight='bold')
plt.imshow(dem_slide, cmap='terrain', vmin=0, vmax=600)
plt.plot([range_section, range_section],[range_span[0], range_span[1]],
         color='r', lw=4, label='Sectional Span')
plt.legend()

plt.subplot(1, 3, 2)
plt.title('Difference Slat Sectional View', fontsize=12, fontweight='bold')
plt.imshow(difference_slant, cmap='coolwarm',)
plt.plot([range_section, range_section],[range_span[0], range_span[1]],
         label='Sectional Span',
         color='g', lw=4 )
plt.legend()

plt.subplot(1, 3, 3)
plt.title('Phase Sectional View', fontsize=12, fontweight='bold')
plt.imshow(ifgm_topo_filtered, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.plot([range_section, range_section],[range_span[0], range_span[1]],
         label='Sectional Span', color='Black', lw=4)

plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_range.png'))
plt.show()
plt.clf();plt.close()

# plot
plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]', )
# plt.ylim(0, 50)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    np.zeros_like(diff_range), 
    -diff_range, 
    color='r', label='Difference', scale_units='xy',
    pivot = 'mid', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')

plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_difference_range.png'))
plt.show();
plt.clf();plt.close()

incident_angle_range_x = np.cos(incident_angle_range)
incident_angle_range_y = -np.sin(incident_angle_range)

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    incident_angle_range_x, 
    incident_angle_range_y, 
    color='g', label='Incident Angle', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_incident_angle_range.png'))
plt.show();
plt.clf();plt.close()

local_incident_angle_range_x, local_incident_angle_range_y = np.cos(np.radians(90 - local_incident_angle_range)), -np.sin(np.radians(local_incident_angle_range))

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    local_incident_angle_range_x, 
    local_incident_angle_range_y, 
    color='b', label='Local Incident Angle', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004,
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_incident_angle_range.png'))
plt.show();
plt.clf();plt.close()

ifgm_range_x, ifgm_range_y = np.cos(ifgm_topo_filtered_range), -np.sin(ifgm_topo_filtered_range)



plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    ifgm_range_x, 
    ifgm_range_y, 
    color='Black', label='Phase', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004,
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_phase_range.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
from windrose import WindroseAxes
from matplotlib import cm

fig = plt.figure(figsize=(12 ,8), facecolor='w', edgecolor='k')

ax = WindroseAxes.from_ax(fig=fig)
ax.bar(np.degrees(ifgm_topo_filtered_range + np.pi) , coh_range, 
       normed=True, opening=0.75, bins=10, edgecolor='g',
       nsector=10, cmap=cm.gnuplot)
ax.set_title('Sectional View Range Phase Direction')

ax.set_rgrids(np.arange(0, 1, 10),labels=list(np.arange(0, 1, 10)),fontsize=8)
ax.set_xticklabels((90, 45, 0, 315, 270, 225, 180, 135))
ax.set_legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topo_direction_range.png'))
plt.show()
plt.clf();

fig = plt.figure(figsize=(12 ,8), facecolor='w', edgecolor='k')

ax = WindroseAxes.from_ax(fig=fig)
ax.contourf(np.degrees(ifgm_topo_filtered_range + np.pi), coh_range, 
       nsector=10, cmap=cm.jet, bins=10)
ax.set_title('Sectional View Range Phase Direction')

ax.set_rgrids(np.arange(0, 1, 10),labels=list(np.arange(0, 1, 10)),fontsize=8)
ax.set_xticklabels((90, 45, 0, 315, 270, 225, 180, 135))
ax.set_legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topo_direction-cont_range.png'))
plt.show()
plt.clf();

In [ ]:
fig = plt.figure(figsize=(12 ,8), facecolor='w', edgecolor='k')

ax = WindroseAxes.from_ax(fig=fig)
ax.bar(np.degrees(ifgm_topo_filtered_line + np.pi), coh_line, 
       normed=True, opening=0.75, bins=10, edgecolor='g',
       nsector=10, cmap=cm.gnuplot)
ax.set_title('Sectional View Line Phase Direction')

ax.set_rgrids(np.arange(0, 1, 10),labels=list(np.arange(0, 1, 10)),fontsize=8)
ax.set_xticklabels((90, 45, 0, 315, 270, 225, 180, 135))
ax.set_legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topo_direction_line.png'))
plt.show()
plt.clf();

In [ ]:
## 
ifgm_rm_orb_range = ifgm_rm_orb[range_span[0]:range_span[1], range_section]

ifgm_orb_range_x = np.cos(ifgm_rm_orb_range)
ifgm_orb_range_y = -np.sin(ifgm_rm_orb_range)


fig = plt.figure(figsize=(12, 8), dpi=120, facecolor='w', edgecolor='k')
ax = WindroseAxes.from_ax(fig=fig)
ax.bar(np.degrees(ifgm_rm_orb_range + np.pi), coh_range, 
       normed=True, opening=0.75, bins=10, edgecolor='g',
       nsector=10, cmap=cm.gnuplot)
ax.set_title('Sectional View Range Phase Direction')

ax.set_rgrids(np.arange(0, 1, 10),labels=list(np.arange(0, 1, 10)),fontsize=8)
ax.set_xticklabels((90, 45, 0, 315, 270, 225, 180, 135))
ax.set_legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_orb_direction_range.png'))
plt.show()
plt.clf();plt.close()


In [ ]:
# sectional view 
range_section = 3000
range_span    = [300, 500]

diff_range = difference_slant[range_span[0]:range_span[1], range_section]
dem_range  = dem_slide[range_span[0]:range_span[1], range_section]
slope_range = slope[range_span[0]:range_span[1], range_section]

incident_angle_range = incident_angle[range_span[0]:range_span[1], range_section]
local_incident_angle_range = local_incident_angle[range_span[0]:range_span[1], range_section]

ifgm_topo_filtered_range = ifgm_topo_filtered[range_span[0]:range_span[1], range_section]
coh_range = coherence[range_span[0]:range_span[1], range_section]

plt.figure(figsize=(18, 18), facecolor='w', edgecolor='k')
plt.subplot(1, 3, 1)
plt.title('DEM Slide Sectional View', fontsize=12, fontweight='bold')
plt.imshow(dem_slide, cmap='terrain', vmin=0, vmax=600)
plt.plot([range_section, range_section],[range_span[0], range_span[1]],
         color='r', lw=4, label='Sectional Span')
plt.legend()

plt.subplot(1, 3, 2)
plt.title('Difference Slat Sectional View', fontsize=12, fontweight='bold')
plt.imshow(difference_slant, cmap='coolwarm',)
plt.plot([range_section, range_section],[range_span[0], range_span[1]],
         label='Sectional Span',
         color='g', lw=4 )
plt.legend()

plt.subplot(1, 3, 3)
plt.title('Phase Sectional View', fontsize=12, fontweight='bold')
plt.imshow(ifgm_topo_filtered, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.plot([range_section, range_section],[range_span[0], range_span[1]],
         label='Sectional Span', color='Black', lw=4)

plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_range_sea.png'))
plt.show()
plt.clf();plt.close()

# plot
plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]', )
# plt.ylim(0, 50)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    np.zeros_like(diff_range), 
    -diff_range, 
    color='r', label='Difference', scale_units='xy',
    pivot = 'mid', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')

plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_difference_range_sea.png'))
plt.show();
plt.clf();plt.close()

incident_angle_range_x = np.cos(incident_angle_range)
incident_angle_range_y = -np.sin(incident_angle_range)

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    incident_angle_range_x, 
    incident_angle_range_y, 
    color='g', label='Incident Angle', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_incident_angle_range_sea.png'))
plt.show();
plt.clf();plt.close()

local_incident_angle_range_x, local_incident_angle_range_y = np.cos(np.radians(90 - local_incident_angle_range)), -np.sin(np.radians(local_incident_angle_range))

plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    local_incident_angle_range_x, 
    local_incident_angle_range_y, 
    color='b', label='Local Incident Angle', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004,
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_incident_angle_range_sea.png'))
plt.show();
plt.clf();plt.close()

ifgm_range_x, ifgm_range_y = np.cos(ifgm_topo_filtered_range), -np.sin(ifgm_topo_filtered_range)



plt.figure(figsize=(12, 4), dpi=120, facecolor='w', edgecolor='k')
plt.plot(dem_range, label='DEM [m]')
# plt.ylim(0, 60)
plt.quiver(
    np.arange(len(diff_range)), 
    dem_range, 
    ifgm_range_x, 
    ifgm_range_y, 
    color='Black', label='Phase', scale_units='xy',
    pivot = 'tail', 
    headwidth=3, headlength=1, headaxislength=0.5, 
    width=0.004,
    )
plt.xlabel('Span [Pixel]')
plt.ylabel('Height [m]')

plt.tight_layout()
plt.legend(loc='upper right')
plt.title('Sectional View', fontsize=12, fontweight='bold')
plt.savefig(os.path.join(PATH_OUTPUT, 'sectional_view_phase_range_sea.png'))
plt.show();
plt.clf();plt.close()

fig = plt.figure(figsize=(12 ,8), facecolor='w', edgecolor='k')

ax = WindroseAxes.from_ax(fig=fig)
ax.bar(np.degrees(ifgm_topo_filtered_range + np.pi), coh_range, 
       normed=True, opening=0.75, bins=10, edgecolor='g',
       nsector=10, cmap=cm.gnuplot)
ax.set_title('Sectional View Range Phase Direction')

ax.set_rgrids(np.arange(0, 1, 10),labels=list(np.arange(0, 1, 10)),fontsize=8)
ax.set_xticklabels((90, 45, 0, 315, 270, 225, 180, 135))
ax.set_legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topo_direction_range_sea.png'))
plt.show()
plt.clf();

fig = plt.figure(figsize=(12 ,8), facecolor='w', edgecolor='k')

ax = WindroseAxes.from_ax(fig=fig)
ax.contourf(np.degrees(ifgm_topo_filtered_range + np.pi), coh_range, 
       nsector=10, cmap=cm.jet, bins=10)
ax.set_title('Sectional View Range Phase Direction')

ax.set_rgrids(np.arange(0, 1, 10),labels=list(np.arange(0, 1, 10)),fontsize=8)
ax.set_xticklabels((90, 45, 0, 315, 270, 225, 180, 135))
ax.set_legend()
plt.savefig(os.path.join(PATH_OUTPUT, 'ifgm_topo_direction-cont_range_sea.png'))
plt.show()
plt.clf();

In [ ]:
bbox

In [ ]:
from whitebox_workflows import download_sample_data, show, WbEnvironment

wbe = WbEnvironment()

swir2, swir1, red = wbe.read_rasters(
    'outputs/001/sentinel_images/S2A_54SVF_20231223_0_L2A_blue_cut.tiff', 
    'outputs/001/sentinel_images/S2A_54SVF_20231223_0_L2A_green_cut.tiff', 
    'outputs/001/sentinel_images/S2A_54SVF_20231223_0_L2A_red_cut.tiff')
image = wbe.create_colour_composite(swir2, swir1, red)

show(image, figsize=(16, 8))

In [ ]:
img_red = tifffile.imread(f'outputs/001/sentinel_images/S2A_54SVF_20231223_0_L2A_red_cut.tiff')
img_green = tifffile.imread(f'outputs/001/sentinel_images/S2A_54SVF_20231223_0_L2A_green_cut.tiff')
img_blue = tifffile.imread(f'outputs/001/sentinel_images/S2A_54SVF_20231223_0_L2A_blue_cut.tiff')

img_rgb = np.stack((to8bit(img_red), to8bit(img_green), to8bit(img_blue)), axis=2)
print(img_rgb.shape)
img_rgb = img_rgb[1500:][::-1,:,:]
plt.imshow(img_rgb)

In [ ]:
from osgeo import gdal, osr

def georeference(img, 
                 LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT, 
                 PATH_GEOTIF, gdal_type=gdal.GDT_Float32, ch_name='Band 1',
                 EPSG=4326, CH=1):
    (H, W) = img.shape[:2]

    driver = gdal.GetDriverByName('GTiff')
    # https://naturalatlas.github.io/node-gdal/classes/Constants%20(GDT).html
    out_dataset = driver.Create(PATH_GEOTIF, W, H, CH, gdal_type)
    
    out_band = out_dataset.GetRasterBand(CH)
    out_band.WriteArray(img)
    out_band.SetDescription(ch_name)
    
    gcp_list = [
        gdal.GCP(LT_LON, LT_LAT, 0, 0, 0),
        gdal.GCP(RT_LON, RT_LAT, 0, W, 0),
        gdal.GCP(RB_LON, RB_LAT, 0, W, H),
        gdal.GCP(LB_LON, LB_LAT, 0, 0, H),
        ]
    
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(EPSG)
    out_dataset.SetGCPs(gcp_list, srs.ExportToWkt())
    out_dataset = None
    
PATH_DINSAR = os.path.join(PATH_OUTPUT, 'dinsar.tif')
LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT = bbox[0], bbox[1], bbox[2], bbox[1], bbox[2], bbox[3], bbox[0], bbox[3]
print(LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT)

georeference(difference_slant[::-1,], 
                 LT_LON, LT_LAT, RT_LON, RT_LAT, RB_LON, RB_LAT, LB_LON, LB_LAT, 
                 PATH_DINSAR, gdal_type=gdal.GDT_Float32, ch_name='Difference Slant',
                 EPSG=4326, CH=1)


In [ ]:
AOI = (100, 1500, 800, 1200) # Y, X

# top n point high coherence
N = 100

coherence_aoi = coherence[AOI[0]:AOI[1], AOI[2]:AOI[3]]

flattened_indices = np.argpartition(coherence_aoi.flatten(), -N)[-N:]
top_n_indices = np.unravel_index(flattened_indices, coherence_aoi.shape)
top_n_values = coherence_aoi[top_n_indices]


dem_slide_aoi = dem_slide[AOI[0]:AOI[1], AOI[2]:AOI[3]]

plt.figure(figsize=(14, 10), dpi=120, facecolor='w', edgecolor='k')
# 3d mesh plot
ax = plt.axes(projection='3d')


x = np.arange(0, dem_slide_aoi.shape[1], 1)
y = np.arange(0, dem_slide_aoi.shape[0], 1)
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, dem_slide_aoi, cmap='ocean_r', edgecolor='none', 
                cstride=1,rstride=1, alpha=0.6,)

# colorbar
fig.colorbar(surf, shrink=0.5)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_zlabel('Elevation [m]')
# ax.set_xlim(0, 1)  # X軸の範囲
# ax.set_ylim(0, 1)  # Y軸の範囲
ax.set_zlim(0, 100)  # Z軸の範囲
ax.view_init(elev=60, azim=-20, roll=0)

diff_dem_aoi = dem_slide[top_n_indices] + difference_slant[top_n_indices]
ax.scatter(top_n_indices[1], top_n_indices[0], diff_dem_aoi, 
             color='r', s=5, label='Hight Coherence', marker='x',
             vmin=0, vmax=100)

plt.legend()

plt.savefig(os.path.join(PATH_OUTPUT, 'dem_slide_3d_difference_point.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
N = 500

coherence_aoi = coherence[AOI[0]:AOI[1], AOI[2]:AOI[3]]

flattened_indices = np.argpartition(coherence_aoi.flatten(), -N)[-N:]
top_n_indices = np.unravel_index(flattened_indices, coherence_aoi.shape)
top_n_values = coherence_aoi[top_n_indices]
diff_dem_aoi = dem_slide[top_n_indices] + difference_slant[top_n_indices]
amp_m_aoi = amp_m_crop[AOI[0]:AOI[1], AOI[2]:AOI[3]]
amp_ms_aoi = amp_ms_crop[AOI[0]:AOI[1], AOI[2]:AOI[3]]
difference_slant_aoi = difference_slant[AOI[0]:AOI[1], AOI[2]:AOI[3]]

In [ ]:
# difference_slant[top_n_indices] を３種類の色で表示
plt.figure(figsize=(28, 12), dpi=160, facecolor='w', edgecolor='k')

plt.subplot(1 ,5, 1)
plt.imshow(dem_slide_aoi, cmap='terrain', vmin=0, vmax=100)
plt.colorbar(shrink=0.25, orientation='horizontal')
plt.scatter(top_n_indices[1], top_n_indices[0], c=difference_slant[top_n_indices], cmap='jet', s=5)
plt.title(f'DEM Slide Top N:{N} High Coherence Points', fontsize=10, fontweight='bold')

plt.subplot(1 ,5, 2)
plt.imshow(amp_m_aoi, cmap='gray', vmin=0,)
plt.colorbar(shrink=0.25, orientation='horizontal')
plt.scatter(top_n_indices[1], top_n_indices[0], c=difference_slant[top_n_indices], cmap='jet', s=6)
plt.title(f'Ampitude Main Top N:{N} High Coherence Points', fontsize=10, fontweight='bold')

plt.subplot(1 ,5, 3)
plt.imshow(amp_ms_aoi,)
plt.colorbar(shrink=0.25, orientation='horizontal')
plt.scatter(top_n_indices[1], top_n_indices[0], c=difference_slant[top_n_indices], cmap='jet', s=6)
plt.title(f'Ampitude Difference Top N:{N} High Coherence Points', fontsize=10, fontweight='bold')

plt.subplot(1 ,5, 4)
plt.imshow(coherence_aoi, cmap='gnuplot', vmin=0, vmax=1)
plt.colorbar(shrink=0.25, orientation='horizontal')
plt.scatter(top_n_indices[1], top_n_indices[0], c=difference_slant[top_n_indices], cmap='jet', s=5)
plt.title(f'Coherence Top N:{N} High Coherence Points', fontsize=10, fontweight='bold')

plt.subplot(1 ,5, 5)
plt.imshow(difference_slant_aoi, cmap='coolwarm', vmin=0, vmax=1)
plt.colorbar(shrink=0.25, orientation='horizontal')
plt.scatter(top_n_indices[1], top_n_indices[0], c=difference_slant[top_n_indices], cmap='jet', s=5)
plt.title(f'Difference Slant Top N:{N} High Coherence Points', fontsize=10, fontweight='bold')


plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, f'difference_top_n{N}_high_coherence.png'))
plt.show();
plt.clf();plt.close()



In [ ]:
difference_slant_aoi[dem_slide_aoi < 0] = 0

fig = plt.figure(figsize=(28, 12), dpi=120, facecolor='w', edgecolor='k')

ax1 = fig.add_subplot(121, projection='3d')
ax1.view_init(elev=60, azim=170, roll=0)

x = np.arange(0, difference_slant_aoi.shape[1], 1)
y = np.arange(0, difference_slant_aoi.shape[0], 1)
X, Y = np.meshgrid(x, y)

surf = ax1.plot_surface(X, Y, difference_slant_aoi, cmap='seismic', edgecolor='none', 
                cstride=1,rstride=1, )

# colorbar
fig.colorbar(surf, shrink=0.5, ax=ax1)

ax1.set_title('Difference Slant 3D')
ax1.set_ylabel('Longitude [Pixel]')
ax1.set_xlabel('Latitude  [Pixel]')
ax1.set_zlabel('Defference Slant [m]')


ax2 = fig.add_subplot(122, projection='3d')
ax2.view_init(elev=60, azim=170, roll=0)

surf = ax2.plot_surface(X, Y, difference_slant_aoi, cmap='hsv', edgecolor='none', 
                cstride=1,rstride=1,)

# colorbar
fig.colorbar(surf, shrink=0.5, ax=ax2)
ax2.set_ylabel('Longitude [Pixel]')
ax2.set_xlabel('Latitude  [Pixel]')
ax2.set_zlabel('Defference Slant [m]')
ax2.set_title('Difference Slant 3D Phase Color')
ax2.set_zlim(-np.pi * wave_length, np.pi * wave_length) 

plt.savefig(os.path.join(PATH_OUTPUT, 'difference_slant_aoi_3D.png'))
plt.show();
plt.clf();plt.close()


In [ ]:
from whitebox_workflows import download_sample_data, show, WbEnvironment, WbPalette

georeference(img=dem_slide[::-1, :],
                LT_LON=bbox[0], LT_LAT=bbox[1], 
                RT_LON=bbox[2], RT_LAT=bbox[1], 
                RB_LON=bbox[2], RB_LAT=bbox[3], 
                LB_LON=bbox[0], LB_LAT=bbox[3], 
                PATH_GEOTIF=os.path.join(PATH_OUTPUT, 'dem_slide.tif'), 
                gdal_type=gdal.GDT_Float32, ch_name='DEM Slide',
                EPSG=4326, CH=1)

wbe = WbEnvironment()

dem = wbe.read_raster(os.path.join(PATH_OUTPUT, 'dem_slide.tif'))

In [ ]:
show(
    dem, 
    figsize=(16, 8),
    skip=1,
    plot_as_surface=True, 
    # vert_exaggeration = 2.5,
    color='lightgreen',
    shade=True,
    linewidth=0.0, 
    rcount=20, # default is 50 rows and columns for grid
    ccount=10,
    antialiased=False
)

In [ ]:
# A mesh grid example
show(dem, plot_as_surface=True, figsize=(16,8), skip=2, 
     alpha=0.0, linewidth=0.75, edgecolors='black')
